In [ ]:
# Run this cell ONCE to set the working directory 
# two levels up to get to the main project folder.
# Then run whichever cells needed below
import os
os.chdir("../..")

# COMPARE ACROSS PROTOCOLS (WITH VLE CALCULATIONS)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# SCOPE CONTROL: 'ALL', 'ASPEN', or 'ML'
PLOT_SCOPE = 'ALL' 

# aths
aspen_root = 'outputs/aspen/binary_results' 
ml_root    = 'outputs/inference/pxy' 

# System Names (File Parsing vs Display)
aspen_sys_files = ('TOLUENE', 'WATER')  
ml_sys_files    = ('Toluene', 'WATER')  
display_names   = ('TOLUENE', 'WATER')  

# available classical models    -> ['WILSON','NRTL' ,'UNIQUAC','UNIFAC','COSMOSAC'] 
# available ML models           -> ['protocol_I','protocol_II','protocol_III','protocol_IV'] 

all_aspen_models = ['WILSON','NRTL','UNIFAC'] 
all_ml_models    = ['protocol_III'] 

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "mathtext.fontset": "stix" 
})
fontsize = 14
temperature = 298.15 

In [ ]:
import numpy as np

if PLOT_SCOPE == 'ASPEN':
    active_aspen = all_aspen_models
    active_ml    = []
    title_suffix = "(Classical Models Only)"
elif PLOT_SCOPE == 'ML':
    active_aspen = []
    active_ml    = all_ml_models
    title_suffix = "(ML Models Only)"
else:
    active_aspen = all_aspen_models
    active_ml    = all_ml_models
    title_suffix = "(Classical and ML)"

title_prefix = f'{display_names[0]}/{display_names[1]} Comparison {title_suffix}'
name1, name2 = display_names

fig, ax = plt.subplots(2, 2, figsize=(16, 13)) 

aspen_styles = [('b', 'o'), ('g', 's'), ('r', '^'), ('k', 'v'), ('m', 'D')]
ml_styles    = [('c', '*'), ('orange', 'X'), ('gray', 'P'), ('brown', 'p')]

legend_handles = []
legend_labels = []

print(f"Generating {PLOT_SCOPE} Comparison for {name1} - {name2}...")

MARKER_INTERVAL = 10

def plot_model_data(model_name, root_path, sys_tuple, color, marker, is_ml=False):
    c1, c2 = sys_tuple
    
    if is_ml:
        sys_folder = f"{c1}_{c2}"
        file_path = os.path.join(root_path, model_name, 'hard', sys_folder, 'vle_data.csv')
    else:
        file_name = f'{c1}_{c2}_{model_name}_vle.csv'
        file_path = os.path.join(root_path, model_name, 'csv', file_name)
    
    if not os.path.exists(file_path):
        print(f"Skipping {model_name}: File not found at {file_path}")
        return

    print(f"Loading {model_name}...")
    df = pd.read_csv(file_path)
    
    lw = 2.5 if is_ml else 2.0

    n_points = len(df)
    markevery = max(1, MARKER_INTERVAL)

    # Plot arrangement: [0,0]=Excess Gibbs, [0,1]=Activity Coeff, [1,0]=Gibbs Mix, [1,1]=VLE
    
    # Excess Gibbs Energy
    ax[0, 0].plot(df['x1'], df['g_excess_reduced'], color=color, marker=marker, markersize=5, 
                  linewidth=lw, markevery=markevery)

    # Activity Coefficients
    line, = ax[0, 1].plot(df['x1'], df['ln_gamma1'], color=color, marker=marker, markersize=5, 
                          linewidth=lw, linestyle='-', markevery=markevery)
    ax[0, 1].plot(df['x1'], df['ln_gamma2'], color=color, marker=marker, markersize=5, 
                  linewidth=lw, linestyle='--', markevery=markevery)
    
    legend_handles.append(line)
    legend_labels.append(model_name)

    # Gibbs Energy of Mixing
    ax[1, 0].plot(df['x1'], df['g_mix_reduced'], color=color, marker=marker, markersize=5, 
                  linewidth=lw, markevery=markevery)

    # VLE Curve (P-x-y)
    ax[1, 1].plot(df['x1'], df['P'], color=color, marker=marker, markersize=5, 
                  linewidth=lw, linestyle='-', markevery=markevery) 
    ax[1, 1].plot(df['y1'], df['P'], color=color, marker=marker, markersize=5, 
                  linewidth=lw, linestyle='-', markevery=markevery)


for i, model in enumerate(active_aspen):
    c, m = aspen_styles[i % len(aspen_styles)]
    plot_model_data(model, aspen_root, aspen_sys_files, c, m, is_ml=False)

for i, model in enumerate(active_ml):
    c, m = ml_styles[i % len(ml_styles)]
    plot_model_data(model, ml_root, ml_sys_files, c, m, is_ml=True)

if not legend_handles:
    print("\nWARNING: No models were plotted! Check paths or model names.")
else:
    for a in ax.flat:
        a.tick_params(axis='both', which='major', labelsize=fontsize)
        a.set_xlim(0, 1)

    ax[0, 0].set_ylabel('$g^E / RT$', fontsize=fontsize)
    ax[0, 0].set_title(f'Excess Gibbs Energy', fontsize=fontsize)
    ax[0, 0].axhline(0, color='black', linewidth=1.0, linestyle='--')
    
    ax[0, 1].set_ylabel('ln $\gamma$', fontsize=fontsize) 
    ax[0, 1].set_title(f'Activity Coefficients', fontsize=fontsize)
    ax[0, 1].axhline(0, color='black', linewidth=1.0, linestyle='--')

    style_text = f'Solid: {name1}\nDashed: {name2}'

    ax_target = ax[0, 1]
    xlim = ax_target.get_xlim()
    ylim = ax_target.get_ylim()
    y_range = ylim[1] - ylim[0] if (ylim[1] - ylim[0]) != 0 else 1.0

    candidates = [
        (0.95, 0.95, 'right', 'top'),
        (0.05, 0.95, 'left', 'top'),
        (0.95, 0.05, 'right', 'bottom'),
        (0.05, 0.05, 'left', 'bottom'),
    ]

    def position_is_clear(ax_obj, frac_x, frac_y, margin_frac=0.08):
        xdata = xlim[0] + frac_x * (xlim[1] - xlim[0])
        ydata = ylim[0] + frac_y * (ylim[1] - ylim[0])
        margin = margin_frac * y_range

        for line in ax_obj.get_lines():
            if not line.get_visible():
                continue
            xd = np.asarray(line.get_xdata())
            yd = np.asarray(line.get_ydata())
            if xd.size == 0 or yd.size == 0:
                continue
            if xdata < xd.min() or xdata > xd.max():
                continue
            try:
                y_at_x = np.interp(xdata, xd, yd)
            except Exception:
                continue
            if np.isnan(y_at_x):
                continue
            if abs(y_at_x - ydata) < margin:
                return False
        return True

    chosen = None
    for fx_c, fy_c, ha_c, va_c in candidates:
        if position_is_clear(ax_target, fx_c, fy_c):
            chosen = (fx_c, fy_c, ha_c, va_c)
            break
    if chosen is None:
        chosen = (0.95, 0.95, 'right', 'top')

    fx, fy, ha_orig, va = chosen

    ha = 'left'
    if ha_orig == 'right':
        fx = max(0.01, fx - 0.28)

    ax_target.text(fx, fy, style_text, transform=ax_target.transAxes,
                   fontsize=max(10, fontsize-2), horizontalalignment=ha, verticalalignment=va,
                   bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="black", alpha=0.85))

    ax[1, 0].set_ylabel('$\Delta_{mix}g / RT$', fontsize=fontsize)
    ax[1, 0].set_title(f'Mixing Gibbs Energy', fontsize=fontsize)
    ax[1, 0].axhline(0, color='black', linewidth=1.0, linestyle='--')

    ax[1, 1].set_ylabel('Pressure (bar)', fontsize=fontsize)
    ax[1, 1].set_title(f'P-x-y Diagram', fontsize=fontsize)

    for a in ax.flat:
        a.set_xlabel(f'mol frac, {name1}', fontsize=fontsize)

    fig.legend(legend_handles, legend_labels, 
               loc='lower center', 
               ncol=min(5, len(legend_handles)),
               fontsize=fontsize, 
               bbox_to_anchor=(0.5, 0.02), 
               frameon=True)

    plt.subplots_adjust(bottom=0.12, hspace=0.3, wspace=0.25)

    # Save
    output_dir = 'outputs/model_comparison/binary_comparison/with_phase_eq'
    os.makedirs(output_dir, exist_ok=True)
    output_file = os.path.join(output_dir, f'{name1}_{name2}_{PLOT_SCOPE}_Comparison.png')
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    print(f"Comparison plot saved as: {output_file}")
    plt.show()

# COMPARE ACROSS MODELS (WITHOUT VLE CALCULATION)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. SCOPE CONTROL: 'ALL', 'ASPEN', or 'ML'
PLOT_SCOPE = 'ALL' 

# 2. Paths
aspen_root = 'outputs/aspen/binary_results' 
ml_root    = 'outputs/inference/pxy' 

# 3. System pairs to loop over
aspen_systems = [('ETHANOL', 'WATER'),
                 ('N-BUTANOL', 'WATER'),
                 ('TOLUENE', 'WATER'),]
ml_systems = [('ETHANOL', 'WATER'),
              ('1-BUTANOL', 'WATER'),
              ('Toluene', 'WATER'),]

# 4. Define the desired models
all_aspen_models = ['UNIFAC','COSMOSAC'] 
all_ml_models    = ['protocol_III'] 

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "mathtext.fontset": "stix" 
})
fontsize = 14
temperature = 298.15 

In [ ]:
if PLOT_SCOPE == 'ASPEN':
    active_aspen = all_aspen_models
    active_ml    = []
    title_suffix = "(Classical Models Only)"
elif PLOT_SCOPE == 'ML':
    active_aspen = []
    active_ml    = all_ml_models
    title_suffix = "(ML Models Only)"
else:
    active_aspen = all_aspen_models
    active_ml    = all_ml_models
    title_suffix = "(Classical and ML)"

aspen_styles = [('brown', 'o'), ('orange', 's'), ('r', '^'), ('k', 'v'), ('m', 'D')]
ml_styles    = [('green', '*'), ('orange', 'X'), ('gray', 'P'), ('brown', 'p')]

MARKER_INTERVAL = 10

def plot_model_data(model_name, root_path, sys_tuple, color, marker, ax_row, is_ml=False):
    c1, c2 = sys_tuple
    
    if is_ml:
        sys_folder = f"{c1}_{c2}"
        file_path = os.path.join(root_path, model_name, 'hard', sys_folder, 'vle_data.csv')
    else:
        file_name = f'{c1}_{c2}_{model_name}_vle.csv'
        file_path = os.path.join(root_path, model_name, 'csv', file_name)
    
    if not os.path.exists(file_path):
        print(f"Skipping {model_name}: File not found at {file_path}")
        return None

    print(f"Loading {model_name}...")
    df = pd.read_csv(file_path)
    
    lw = 2.5 if is_ml else 2.0
    markevery = max(1, MARKER_INTERVAL)

    
    # Excess Gibbs Energy
    ax_row[0].plot(df['x1'], df['g_excess_reduced'], color=color, marker=marker, markersize=5, 
                  linewidth=lw, markevery=markevery)

    # Activity Coefficients
    line, = ax_row[1].plot(df['x1'], df['ln_gamma1'], color=color, marker=marker, markersize=5, 
                          linewidth=lw, linestyle='-', markevery=markevery)
    ax_row[1].plot(df['x1'], df['ln_gamma2'], color=color, marker=marker, markersize=5, 
                  linewidth=lw, linestyle='--', markevery=markevery)

    # Gibbs Energy of Mixing
    ax_row[2].plot(df['x1'], df['g_mix_reduced'], color=color, marker=marker, markersize=5, 
                  linewidth=lw, markevery=markevery)
    
    return line

fig, ax = plt.subplots(3, 3, figsize=(18, 14))

legend_handles = []
legend_labels = []

for sys_idx, (aspen_sys, ml_sys) in enumerate(zip(aspen_systems, ml_systems)):
    name1, name2 = aspen_sys
    print(f"\nGenerating comparison for {name1}/{name2}...")
    
    ax_row = ax[sys_idx, :]
    
    for i, model in enumerate(active_aspen):
        c, m = aspen_styles[i % len(aspen_styles)]
        line = plot_model_data(model, aspen_root, aspen_sys, c, m, ax_row, is_ml=False)
        if line and sys_idx == 0:
            legend_handles.append(line)
            legend_labels.append(model)

    for i, model in enumerate(active_ml):
        c, m = ml_styles[i % len(ml_styles)]
        line = plot_model_data(model, ml_root, ml_sys, c, m, ax_row, is_ml=True)
        if line and sys_idx == 0:
            legend_handles.append(line)
            legend_labels.append(model)
    
    for col in range(3):
        ax_row[col].tick_params(axis='both', which='major', labelsize=fontsize)
        ax_row[col].set_xlim(0, 1)
        ax_row[col].axhline(0, color='black', linewidth=1.0, linestyle='--')
        ax_row[col].set_xlabel(f'mol frac, {name1}', fontsize=fontsize)
    
    ax_row[0].set_ylabel('$g^E / RT$', fontsize=fontsize)
    ax_row[0].set_title(f'{name1}/{name2} - Excess Gibbs Energy', fontsize=fontsize)
    
    ax_row[1].set_ylabel('ln $\gamma$', fontsize=fontsize)
    ax_row[1].set_title(f'{name1}/{name2} - Activity Coefficients', fontsize=fontsize)
    
    ax_row[2].set_ylabel('$\Delta_{mix}g / RT$', fontsize=fontsize)
    ax_row[2].set_title(f'{name1}/{name2} - Mixing Gibbs Energy', fontsize=fontsize)

if legend_handles:
    fig.legend(legend_handles, legend_labels, 
               loc='lower center', 
               ncol=min(5, len(legend_handles)),
               fontsize=fontsize, 
               bbox_to_anchor=(0.5, 0.02), 
               frameon=True)

plt.subplots_adjust(bottom=0.12, hspace=0.4, wspace=0.3)

output_dir = 'outputs/model_comparison/binary_comparison/no_phase_eq'
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, f'Systems_Comparison_{PLOT_SCOPE}.png')
plt.savefig(output_file, dpi=300, bbox_inches='tight')
print(f"\nComparison plot saved as: {output_file}")
plt.show()


# COMPARE TERNARY SURFACE

In [ ]:
# Plot ternary gibbs energy analysis across UNIFAC, COSMO-SAC, PROTOCOL III
# The CSV is formatted as such -> x1,x2,x3,ln_gamma1,ln_gamma2,ln_gamma3,g_mix_reduced,g_excess_reduced
import mpltern
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import os

aspen_ternary_root = 'outputs/aspen/ternary_results'
ml_ternary_root = 'outputs/inference/ternary'

ternary_systems = [
    ('CHLOROFORM', 'ACETONE', 'METHANOL')
]

ternary_aspen_models = ['UNIFAC', 'COSMO-SAC']
ternary_ml_models = ['protocol_IV']

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "mathtext.fontset": "stix"
})
fontsize = 14

In [ ]:
def apply_ternary_labels(ax, name_t, name_l, name_r, off=0.16):
    ax.set_tlabel('')
    ax.set_llabel('')
    ax.set_rlabel('')
    
    # Top variable (x1)
    ax.text(-off, 0.5 + off/2, 0.5 + off/2, name_t, 
            fontsize=fontsize-2, ha='center', va='top', rotation=0)
    
    # Left variable (x2)
    ax.text(0.5 + off/2, 0.5 + off/2, -off, name_l, 
            fontsize=fontsize-2, ha='center', va='bottom', rotation=60)
    
    # Right variable (x3)
    ax.text(0.5 + off/2, -off, 0.5 + off/2, name_r, 
            fontsize=fontsize-2, ha='center', va='bottom', rotation=-60)

for sys_idx, system in enumerate(ternary_systems):
    name1, name2, name3 = system
    print(f"\nGenerating ternary comparison for {name1}/{name2}/{name3}...")
    
    all_models = ternary_aspen_models + ternary_ml_models
    n_models = len(all_models)
    
    fig = plt.figure(figsize=(18, 14))
    
    g_excess_vals = []
    g_mix_vals = []
    model_data = {} 
    
    for model in all_models:
        if model in ternary_aspen_models:
            file_path = f'{aspen_ternary_root}/{model}/csv/{name1}_{name2}_{name3}_ternary.csv'
        else:
            file_path = f'{ml_ternary_root}/{model}/hard/{name1}_{name2}_{name3}.csv'
            
        if os.path.exists(file_path):
            df = pd.read_csv(file_path)
            model_data[model] = df
            g_excess_vals.extend(df['g_excess_reduced'].values)
            g_mix_vals.extend(df['g_mix_reduced'].values)
        else:
            print(f"Warning: File not found for {model} at {file_path}")
            model_data[model] = None

    # Use actual data min/max ("based on energies") with small buffer
    def get_tight_limits(vals):
        if not vals: return -0.1, 0.1
        v_min, v_max = np.min(vals), np.max(vals)
        span = v_max - v_min
        return v_min - 0.02*span, v_max + 0.02*span

    vmin_excess, vmax_excess = get_tight_limits(g_excess_vals)
    vmin_mix, vmax_mix = get_tight_limits(g_mix_vals)

    levels_excess = np.linspace(vmin_excess, vmax_excess, 41)
    levels_mix = np.linspace(vmin_mix, vmax_mix, 41)

    # Plotting
    # Global layout settings used to calculate colorbar width
    plot_left = 0.1
    plot_right = 0.9
    
    plt.subplots_adjust(left=plot_left, right=plot_right, top=0.92, bottom=0.15, wspace=0.45, hspace=0.6)

    for model_idx, model in enumerate(all_models):
        df = model_data[model]
        if df is None:
            continue
        
        # Top Row: g_excess
        ax_excess = fig.add_subplot(2, n_models, model_idx + 1, projection='ternary')
        ax_excess.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0)
        
        ax_excess.tricontourf(df['x1'], df['x2'], df['x3'], df['g_excess_reduced'], 
                              levels=levels_excess, cmap='RdBu_r', extend='both')
        ax_excess.tricontour(df['x1'], df['x2'], df['x3'], df['g_excess_reduced'], 
                             levels=levels_excess, colors='k', linewidths=0.2, alpha=1)
        
        apply_ternary_labels(ax_excess, name1, name2, name3)
        ax_excess.set_title(model, fontsize=fontsize, pad=35, weight='bold')

        # Bottom Row: g_mix
        ax_mix = fig.add_subplot(2, n_models, n_models + model_idx + 1, projection='ternary')
        ax_mix.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0)
        
        ax_mix.tricontourf(df['x1'], df['x2'], df['x3'], df['g_mix_reduced'], 
                           levels=levels_mix, cmap='RdBu_r', extend='both')
        ax_mix.tricontour(df['x1'], df['x2'], df['x3'], df['g_mix_reduced'], 
                          levels=levels_mix, colors='k', linewidths=0.2, alpha=1)
        
        apply_ternary_labels(ax_mix, name1, name2, name3)

    #  Colorbars 
    
    cbar_width = plot_right - plot_left
    
    norm_excess = mcolors.Normalize(vmin=vmin_excess, vmax=vmax_excess)
    sm_excess = cm.ScalarMappable(cmap='RdBu_r', norm=norm_excess)
    sm_excess.set_array([])
    
    cax_excess = fig.add_axes([plot_left, 0.58, cbar_width, 0.02]) 
    cbar_excess = fig.colorbar(sm_excess, cax=cax_excess, orientation='horizontal')
    cbar_excess.set_label('Excess Gibbs Energy', fontsize=fontsize+2, labelpad=10)

    norm_mix = mcolors.Normalize(vmin=vmin_mix, vmax=vmax_mix)
    sm_mix = cm.ScalarMappable(cmap='RdBu_r', norm=norm_mix)
    sm_mix.set_array([])
    
    cax_mix = fig.add_axes([plot_left, 0.1, cbar_width, 0.02]) 
    cbar_mix = fig.colorbar(sm_mix, cax=cax_mix, orientation='horizontal')
    cbar_mix.set_label('Gibbs Energy of Mixing', fontsize=fontsize+2, labelpad=10)

    
    output_dir = 'outputs/model_comparison/ternary_comparison'
    os.makedirs(output_dir, exist_ok=True)
    output_file = f'{output_dir}/{name1}_{name2}_{name3}_Ternary_Comparison_Final.png'
    
    plt.savefig(output_file, dpi=300) 
    print(f"Ternary plot saved as: {output_file}")
    plt.show()